# SecureRAG


In [2]:
# Env Setup
import os
import sys

rootpath = "/home/huahua/Projects/transformers-3.3.1"
sys.path.insert(0, os.path.join(rootpath + "/" + "src"))

# Logging
import logging

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

file_handler = logging.FileHandler(rootpath + "/" + "tmp/secure_rag.log", mode="a", encoding="utf-8")
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))

console_handler = logging.StreamHandler()
logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")

logger.addHandler(file_handler)
logger.addHandler(console_handler)

## Model Define

In [3]:
# ConfidentialRagRetriever
import random
from typing import List

import numpy as np
from transformers.tokenization_utils_base import BatchEncoding
from transformers.retrieval_rag import RagRetriever


class ConfidentialRagRetriever(RagRetriever):
    def __init__(
        self,
        config,
        question_encoder_tokenizer,
        generator_tokenizer,
        index=None,
    ):
        super().__init__(
            config, question_encoder_tokenizer=question_encoder_tokenizer, generator_tokenizer=generator_tokenizer
        )
        if config is not None:
            self.config = config
            self.n_docs = config.n_docs
            self.batch_size = config.retrieval_batch_size

        if generator_tokenizer is not None:
            self.generator_tokenizer = generator_tokenizer
        if question_encoder_tokenizer is not None:
            self.question_encoder_tokenizer = question_encoder_tokenizer

        if index is not None:
            self.index = index

    def concatConfidentialDocs(self, inputIds, questionIputIds, confidentialDocs, return_tensors=None):
        def cat_input_and_doc(doc_title, doc_text, question=""):
            if doc_title.startswith('"'):
                doc_title = doc_title[1:]
            if doc_title.endswith('"'):
                doc_title = doc_title[:-1]
            out = (" " + doc_title + self.config.title_sep + doc_text + self.config.doc_sep + question).replace(
                "  ", " "
            )
            return out

        questionInputStrings = self.question_encoder_tokenizer.batch_decode(questionIputIds, skip_special_tokens=True)

        confidentialRagInputStrings = [
            cat_input_and_doc(
                doc_title=confidentialDocs[i]["title"][j],
                doc_text=confidentialDocs[i]["text"][j],
                question=questionInputStrings[i] if len(questionInputStrings) > 0 else "",
            )
            for i in range(len(confidentialDocs))
            for j in range(len(confidentialDocs[0]["title"]))
        ]
        contextualized_inputs = self.generator_tokenizer.batch_encode_plus(
            confidentialRagInputStrings,
            max_length=self.config.max_combined_length,
            return_tensors=return_tensors,
            padding="max_length",
            truncation=True,
        )
        return BatchEncoding(
            {
                "input_ids": contextualized_inputs["input_ids"],
                "attention_mask": contextualized_inputs["attention_mask"],
            },
            tensor_type=return_tensors,
        )

    def postprocess_docs(self, docs, input_strings, prefix, n_docs, return_tensors=None):
        def cat_input_and_doc(doc_title, doc_text, input_string, prefix):
            # TODO(Patrick): if we train more RAG models, I want to put the input first to take advantage of effortless truncation
            # TODO(piktus): better handling of truncation
            if doc_title.startswith('"'):
                doc_title = doc_title[1:]
            if doc_title.endswith('"'):
                doc_title = doc_title[:-1]
            if prefix is None:
                prefix = ""
            out = (prefix + doc_title + self.config.title_sep + doc_text + self.config.doc_sep + input_string).replace(
                "  ", " "
            )
            return out

        ctxInputStrings = [
            cat_input_and_doc(
                docs[i]["title"][j],
                docs[i]["text"][j],
                input_strings[i],
                prefix,
            )
            for i in range(len(docs))
            for j in range(n_docs)
        ]

        contextualized_inputs = self.generator_tokenizer.batch_encode_plus(
            ctxInputStrings,
            max_length=self.config.max_combined_length,
            return_tensors=return_tensors,
            padding="max_length",
            truncation=True,
        )

        return (
            contextualized_inputs["input_ids"],
            contextualized_inputs["attention_mask"],
        )

    def __call__(
        self,
        question_input_ids: List[List[int]],
        question_hidden_states: np.ndarray,
        prefix=None,
        n_docs=None,
        numConfidential=0,
        return_tensors=None,
    ) -> BatchEncoding:
        n_docs = n_docs if n_docs is not None else self.n_docs
        prefix = prefix if prefix is not None else self.config.generator.prefix
        retrieved_doc_embeds, doc_ids, docs = self.retrieve(question_hidden_states, n_docs + numConfidential)

        input_strings = self.question_encoder_tokenizer.batch_decode(question_input_ids, skip_special_tokens=True)

        normalDocs = []
        confidentialDocs = []
        for doc in docs:
            title2text = list(zip(doc["title"], doc["text"]))
            random.shuffle(title2text)
            normalDocs.append(
                {
                    "title": [item[0] for item in title2text[:n_docs]],
                    "text": [item[1] for item in title2text[:n_docs]],
                }
            )
            confidentialDocs.append(
                {
                    "title": [item[0] for item in title2text[n_docs:]],
                    "text": [item[1] for item in title2text[n_docs:]],
                }
            )

        context_input_ids, context_attention_mask = self.postprocess_docs(
            normalDocs, input_strings, prefix, n_docs, return_tensors=return_tensors
        )
        return (
            BatchEncoding(
                {
                    "context_input_ids": context_input_ids,
                    "context_attention_mask": context_attention_mask,
                    "retrieved_doc_embeds": retrieved_doc_embeds[:, :n_docs],
                    "doc_ids": doc_ids[:, :n_docs],
                },
                tensor_type=return_tensors,
            ),
            confidentialDocs,
        )

Environment variable FAISS_OPT_LEVEL is not set, so let's pick the instruction set according to the current CPU
Loading faiss with AVX512 support.
Successfully loaded faiss with AVX512 support.


In [4]:
# ConfidentialRagModel
from typing import Optional, Tuple, Union
from transformers.modeling_rag import RagModel, RetrievAugLMOutput
from transformers import PretrainedConfig, PreTrainedModel
import torch


class ConfidentialRagModel(RagModel):
    def __init__(
        self,
        config: Optional[PretrainedConfig] = None,
        question_encoder: Optional[PreTrainedModel] = None,
        generator: Optional[PreTrainedModel] = None,
        retriever: Optional = None,  # or maybe just use a `set_retriever(...)` method # type: ignore
        **kwargs,
    ):
        super().__init__(
            config=config,
            question_encoder=question_encoder,
            generator=generator,
            retriever=retriever,
            kwargs=kwargs,
        )
        self.numConfidential = 0

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        encoder_outputs: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
        decoder_input_ids: Optional[torch.LongTensor] = None,
        decoder_attention_mask: Optional[torch.BoolTensor] = None,
        past_key_values: Optional[Tuple[Tuple[torch.FloatTensor]]] = None,
        doc_scores: Optional[torch.FloatTensor] = None,
        context_input_ids: Optional[torch.LongTensor] = None,
        context_attention_mask: Optional[torch.LongTensor] = None,
        use_cache: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        output_retrieved: Optional[bool] = None,
        n_docs: Optional[int] = None,
    ) -> Union[Tuple[torch.Tensor], RetrievAugLMOutput]:
        n_docs = n_docs if n_docs is not None else self.config.n_docs
        # n_docs = n_docs - self.numConfidential
        use_cache = use_cache if use_cache is not None else self.config.use_cache
        output_attentions = (
            output_attentions
            if output_attentions is not None
            else self.config.output_attentions
        )
        output_hidden_states = (
            output_hidden_states
            if output_hidden_states is not None
            else self.config.output_hidden_states
        )
        output_retrieved = (
            output_retrieved
            if output_retrieved is not None
            else self.config.output_retrieved
        )

        # whether retriever has to be used
        has_to_retrieve = (
            self.retriever is not None
            and (
                context_input_ids is None
                or context_attention_mask is None
                or doc_scores is None
            )
            and encoder_outputs is None
        )
        # encoder_outputs are pre-computed during RAG-token generation
        if encoder_outputs is None:
            if has_to_retrieve:
                question_enc_outputs = self.question_encoder(
                    input_ids, attention_mask=attention_mask, return_dict=True
                )
                question_encoder_last_hidden_state = question_enc_outputs[
                    0
                ]  # hidden states of question encoder

                retriever_outputs = self.retriever(
                    input_ids,
                    question_encoder_last_hidden_state.cpu()
                    .detach()
                    .to(torch.float32)
                    .numpy(),
                    prefix=self.generator.config.prefix,
                    n_docs=n_docs,
                    return_tensors="pt",
                )[0]

                (
                    context_input_ids,
                    context_attention_mask,
                    retrieved_doc_embeds,
                    retrieved_doc_ids,
                ) = (
                    retriever_outputs["context_input_ids"],
                    retriever_outputs["context_attention_mask"],
                    retriever_outputs["retrieved_doc_embeds"],
                    retriever_outputs["doc_ids"],
                )

                # set to correct device
                retrieved_doc_embeds = retrieved_doc_embeds.to(
                    question_encoder_last_hidden_state
                )
                context_input_ids = context_input_ids.to(input_ids)
                context_attention_mask = context_attention_mask.to(input_ids)

                # compute doc_scores
                doc_scores = torch.bmm(
                    question_encoder_last_hidden_state.unsqueeze(1),
                    retrieved_doc_embeds.transpose(1, 2),
                ).squeeze(1)
            else:
                assert context_input_ids is not None, (
                    "Make sure that `context_input_ids` are passed, if no `retriever` is set. Alternatively, you can"
                    " set a retriever using the `set_retriever(...)` function."
                )
                assert context_attention_mask is not None, (
                    "Make sure that `context_attention_mask` are passed, if no `retriever` is set. Alternatively, you"
                    " can set a retriever using the `set_retriever(...)` function."
                )
                assert doc_scores is not None, (
                    "Make sure that `doc_scores` are passed, if no `retriever` is set. Alternatively, you can set a"
                    " retriever using the `set_retriever(...)` function."
                )

        assert (
            doc_scores is not None
        ), "Make sure that `doc_scores` are passed when passing `encoder_outputs` to the forward function."

        assert (doc_scores.shape[1] % n_docs) == 0, (
            f" The first dimension of `context_input_ids` should be a multiple of `n_docs`={n_docs}, but is"
            f" {context_input_ids.shape[0]}."
        )

        # Decoder input without context documents
        if decoder_input_ids is not None:
            decoder_input_ids = decoder_input_ids.repeat_interleave(n_docs, dim=0)

        if decoder_attention_mask is not None:
            decoder_attention_mask = decoder_attention_mask.repeat_interleave(
                n_docs, dim=0
            )

        gen_outputs = self.generator(
            input_ids=context_input_ids,
            attention_mask=context_attention_mask,
            encoder_outputs=encoder_outputs,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_attentions=output_attentions,
            return_dict=True,
        )

        if not has_to_retrieve:
            question_encoder_last_hidden_state = None
            question_enc_hidden_states = None
            question_enc_attentions = None
            retrieved_doc_embeds = None
            retrieved_doc_ids = None
        else:
            question_enc_hidden_states = question_enc_outputs.hidden_states
            question_enc_attentions = question_enc_outputs.attentions

        if not has_to_retrieve or not output_retrieved:
            # don't output retrieved docs
            context_input_ids = (None,)
            context_attention_mask = None
            retrieved_doc_embeds = None
            retrieved_doc_ids = None

        return RetrievAugLMOutput(
            logits=gen_outputs.logits,
            doc_scores=doc_scores,
            past_key_values=gen_outputs.past_key_values,
            context_input_ids=context_input_ids,
            context_attention_mask=context_attention_mask,
            retrieved_doc_embeds=retrieved_doc_embeds,
            retrieved_doc_ids=retrieved_doc_ids,
            question_encoder_last_hidden_state=question_encoder_last_hidden_state,
            question_enc_hidden_states=question_enc_hidden_states,
            question_enc_attentions=question_enc_attentions,
            generator_enc_last_hidden_state=gen_outputs.encoder_last_hidden_state,
            generator_enc_hidden_states=gen_outputs.encoder_hidden_states,
            generator_enc_attentions=gen_outputs.encoder_attentions,
            generator_dec_hidden_states=gen_outputs.decoder_hidden_states,
            generator_dec_attentions=gen_outputs.decoder_attentions,
        )

In [5]:
# ConfidentialRagSequenceForGeneration

from concurrent.futures import ThreadPoolExecutor
from typing import Optional
from transformers.configuration_rag import RagConfig
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers.modeling_rag import RagSequenceForGeneration
from transformers.retrieval_rag import RagRetriever

import torch


class ConfidentialRagSequenceForGeneration(RagSequenceForGeneration):
    def __init__(
        self,
        config: Optional[PretrainedConfig] = None,
        question_encoder: Optional[PreTrainedModel] = None,
        generator: Optional[PreTrainedModel] = None,
        retriever: Optional[RagRetriever] = None,
        **kwargs,
    ):
        assert config is not None or (
            question_encoder is not None and generator is not None
        ), "Either a configuration or an encoder and a generator has to be provided."

        if config is None:
            config = RagConfig.from_question_encoder_generator_configs(
                question_encoder.config, generator.config, **kwargs
            )
        super().__init__(config)
        self.numConfidential = 0
        self.generationMode = "normalConfidential"
        self.normalOutputTopK = 1

        # instantiate model
        self.rag = ConfidentialRagModel(
            config=config,
            question_encoder=self.rag.question_encoder,
            generator=self.rag.generator,
            retriever=retriever,
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.LongTensor] = None,
        ctxInputIds: Optional[torch.LongTensor] = None,
        context_attention_mask: Optional[torch.LongTensor] = None,
        doc_scores: Optional[torch.FloatTensor] = None,
        do_deduplication: Optional[bool] = None,  # defaults to True
        num_return_sequences: Optional[int] = None,  # defaults to 1
        num_beams: Optional[int] = None,  # defaults to 1
        n_docs: Optional[int] = None,
        **model_kwargs,
    ) -> torch.LongTensor:
        n_docs = n_docs if n_docs is not None else self.config.n_docs
        n_docs = n_docs - self.numConfidential
        do_deduplication = do_deduplication if do_deduplication is not None else self.config.do_deduplication
        num_doc_return_sequences = (
            num_return_sequences if num_return_sequences is not None else self.config.num_return_sequences
        )
        num_beams = num_beams if num_beams is not None else self.config.num_beams
        assert (
            input_ids is not None or ctxInputIds is not None
        ), " At least one of input_ids or context_input_ids must be given"

        def doNormalConfidentialGeneration():
            if self.retriever is not None:
                question_hidden_states = self.question_encoder(input_ids, attention_mask=attention_mask)[0]
                retrievalResult = self.retriever(
                    input_ids,
                    question_hidden_states.cpu().detach().to(torch.float32).numpy(),
                    prefix=self.generator.config.prefix,
                    n_docs=n_docs,
                    numConfidential=self.numConfidential,
                    return_tensors="pt",
                )

                ctxInputIds = retrievalResult[0]["context_input_ids"]
                confidentialDocs = retrievalResult[1]

                # set to correct device
                ctxInputIds = ctxInputIds.to(input_ids)

            hypos = []
            model_kwargs["num_beams"] = num_beams
            model_kwargs["num_return_sequences"] = num_beams
            model_kwargs["attention_mask"] = None

            batch_size = input_ids.shape[0] if input_ids is not None else ctxInputIds.shape[0] // n_docs

            for index in range(batch_size):
                # MARK: normal generation stage.
                # first, generate beams from documents:
                generator_input_ids = ctxInputIds[index * n_docs : (index + 1) * n_docs]  # (n_docs, max_len)

                output_sequences = self.generator.generate(
                    generator_input_ids,
                    **model_kwargs,
                )  # n_docs * n_beam, tgt_len
                if do_deduplication:
                    # do_deduplication, max_output_len
                    output_sequences = torch.stack(list({str(k.tolist()): k for k in output_sequences}.values()))

                # after deduplication, this number can be less than n_docs*n_beam
                num_candidates = output_sequences.shape[0]

                # then, run model forwards to get nll scores:
                if input_ids is not None:
                    new_input_ids = input_ids[index : index + 1].repeat(num_candidates, 1)
                    outputs = self(new_input_ids, labels=output_sequences, exclude_bos_score=True)
                else:  # input_ids is None, need context_input_ids/mask and doc_scores
                    assert context_attention_mask is not None, (
                        "Make sure that `context_attention_mask` are passed, if no `input_ids` is set. Alternatively, you"
                        " can set a retriever using the `set_retriever(...)` function."
                    )
                    assert doc_scores is not None, (
                        "Make sure that `doc_scores` are passed, if no `input_ids` is set. Alternatively, you can set a"
                        " retriever using the `set_retriever(...)` function."
                    )

                    individual_input_ids = generator_input_ids.repeat(
                        num_candidates, 1
                    )  # (num_candidates*n_docs, max_len)

                    individual_attention_mask = context_attention_mask[index * n_docs : (index + 1) * n_docs]
                    individual_attention_mask = individual_attention_mask.repeat(num_candidates, 1)

                    individual_doc_scores = doc_scores[index : (index + 1), :]  # doc_scores.shape = [batch, n_docs]
                    individual_doc_scores = individual_doc_scores.repeat(num_candidates, 1)  # [num_candidates, n_docs]

                    outputs = self(
                        context_input_ids=individual_input_ids,
                        context_attention_mask=individual_attention_mask,
                        doc_scores=individual_doc_scores,
                        labels=output_sequences,
                        exclude_bos_score=True,
                    )

                normalOutputTopK = self.normalOutputTopK
                top_cand_inds = (-outputs["loss"]).topk(normalOutputTopK)[1]

                # MARK: confidential generation stage.
                confidentialDoc = confidentialDocs[index]
                confidentialRagInputIds = self.retriever.concatConfidentialDocs(
                    inputIds=output_sequences[top_cand_inds],
                    questionIputIds=input_ids[index : index + 1],
                    confidentialDocs=[confidentialDoc for _ in range(normalOutputTopK)],
                    return_tensors="pt",
                )["input_ids"]
                confidentialRagInputIds = confidentialRagInputIds.to(input_ids)
                confidentialOutputSeqence = self.generator.generate(
                    confidentialRagInputIds,
                    **model_kwargs,
                )
                if do_deduplication:
                    # do_deduplication, max_output_len
                    confidentialOutputSeqence = torch.stack(
                        list({str(k.tolist()): k for k in confidentialOutputSeqence}.values())
                    )

                num_candidates = confidentialOutputSeqence.shape[0]
                new_input_ids = input_ids[index : index + 1].repeat(num_candidates, 1)
                finalOutputs = self(
                    new_input_ids,
                    labels=confidentialOutputSeqence,
                    exclude_bos_score=True,
                )
                top_cand_inds = (-finalOutputs["loss"]).topk(num_doc_return_sequences)[1]

                hypos.append(confidentialOutputSeqence[top_cand_inds])

            return self._cat_and_pad(hypos, pad_token_id=self.config.generator.pad_token_id)

        def doParallelSummaryGeneration():
            ctxNormalInputIds = ctxInputIds
            if self.retriever is not None and ctxNormalInputIds is None:
                question_hidden_states = self.question_encoder(input_ids, attention_mask=attention_mask)[0]
                retrievalResult = self.retriever(
                    input_ids,
                    question_hidden_states.cpu().detach().to(torch.float32).numpy(),
                    prefix=self.generator.config.prefix,
                    n_docs=n_docs,
                    numConfidential=self.numConfidential,
                    return_tensors="pt",
                )

                ctxNormalInputIds = retrievalResult[0]["context_input_ids"]

                confidentialDocs = retrievalResult[1]
                ctxConfidentialInputIds = self.retriever.concatConfidentialDocs(
                    inputIds="",
                    questionIputIds=input_ids,
                    confidentialDocs=confidentialDocs,
                    return_tensors="pt",
                )["input_ids"]
                # set to correct device
                ctxNormalInputIds = ctxNormalInputIds.to(input_ids)
                ctxConfidentialInputIds = ctxConfidentialInputIds.to(input_ids)

            hypos = []
            model_kwargs["num_beams"] = num_beams
            model_kwargs["num_return_sequences"] = num_beams
            model_kwargs["attention_mask"] = None

            batch_size = input_ids.shape[0] if input_ids is not None else ctxNormalInputIds.shape[0] // n_docs

            executor = ThreadPoolExecutor(max_workers=3)
            for index in range(batch_size):
                # MARK: PARALLEL STAGE
                def doParallelGenerate(inputIds):
                    def doGenerate(ids):
                        output = self.generator.generate(
                            ids,
                            **model_kwargs,
                        )
                        return output

                    result = executor.submit(doGenerate, inputIds)
                    return result

                normalInputIds = ctxNormalInputIds[index * n_docs : (index + 1) * n_docs]
                confidentialInputIds = ctxConfidentialInputIds[
                    index * self.numConfidential : (index + 1) * self.numConfidential
                ]
                print(
                    f"normal input ids is {self.retriever.generator_tokenizer.batch_decode(normalInputIds, skip_special_tokens=True)}"
                )
                print(
                    f"confidential input ids is {self.retriever.generator_tokenizer.batch_decode(confidentialInputIds, skip_special_tokens=True)}"
                )
                # confidentialInputShape = confidentialInputIds.shape
                # confidentialInputIds = confidentialInputIds.view(
                #     -1, confidentialInputShape[0] * confidentialInputShape[1]
                # )
                results = [doParallelGenerate(ids) for ids in [normalInputIds, confidentialInputIds]]
                normalCandidates, confidentialCandidates = (re.result() for re in results)

                # MARK: SUMMARY STAGE
                def combineCandidates(a, b):
                    long, short = (a, b) if a.shape[-1] > b.shape[-1] else (b, a)
                    padid = self.retriever.generator_tokenizer.pad_token_id
                    short = torch.nn.functional.pad(
                        short,
                        (0, long.shape[-1] - short.shape[-1], 0, 0),
                        mode="constant",
                        value=padid,
                    )
                    return torch.cat((long, short))

                candidates = combineCandidates(normalCandidates, confidentialCandidates)
                print(
                    f"candidates is {self.retriever.generator_tokenizer.batch_decode(candidates, skip_special_tokens=True)}"
                )
                if do_deduplication:
                    candidates = torch.stack(list({str(k.tolist()): k for k in candidates}.values()))
                num_candidates = candidates.shape[0]
                new_input_ids = input_ids[index : index + 1].repeat(num_candidates, 1)
                outputs = self(new_input_ids, labels=candidates, exclude_bos_score=True)
                top_cand_inds = (-outputs["loss"]).topk(num_doc_return_sequences)[1]
                hypos.append(candidates[top_cand_inds])

            executor.shutdown(wait=True)
            return self._cat_and_pad(hypos, pad_token_id=self.config.generator.pad_token_id)

        if self.generationMode == "normalConfidential":
            return doNormalConfidentialGeneration()
        elif self.generationMode == "parallelSummary":
            return doParallelSummaryGeneration()
        else:  # default mode.
            return doParallelSummaryGeneration()

## Model Construction

In [6]:
# Global variable
import argparse
import torch

args = argparse.Namespace()
args.model_name_or_path = rootpath + "/" + "models/rag-sequence-nq"
args.model_type = "confidential_rag_sequence"
args.evaluation_set = rootpath + "/" + "examples/rag/output/biencoder-nq-dev.questions"
args.gold_data_path = rootpath + "/" + "examples/rag/output/biencoder-nq-dev.ans"
args.predictions_path = rootpath + "/" + "examples/rag/output/e2e_preds.txt"
args.gold_data_mode = "ans"
args.eval_mode = "e2e"
args.n_docs = 10
args.index_name = None
args.index_path = None
args.k = 1
args.eval_all_checkpoints = False
args.eval_batch_size = 4
args.print_predictions = True
args.recalculate = True
args.num_beams = 4
args.min_length = 1
args.max_length = 50
args.print_docs = False
args.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
args.numConfidential = 5
args.generationMode = "parallelSummary"

In [8]:
# Confidential Model

import ast
import logging
import os
import sys

import pandas as pd
import torch
from tqdm import tqdm

from transformers import (
    BartForConditionalGeneration,
    RagRetriever,
    RagSequenceForGeneration,
    RagTokenForGeneration,
)
from transformers import logging as transformers_logging


rag_relative_path = "examples/rag"
sys.path.append(os.path.join(os.getcwd() + "/" + rag_relative_path))  # noqa: E402 # isort:skip

sys.path.append(os.path.join(os.getcwd()))  # noqa: E402 # isort:skip
# from examples.rag.utils import exact_match_score, f1_score  # noqa: E402 # isort:skip
from utils import exact_match_score, f1_score  # noqa: E402 # isort:skip


logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

transformers_logging.set_verbosity_info()


def infer_model_type(model_name_or_path):
    if "token" in model_name_or_path:
        return "rag_token"
    if "sequence" in model_name_or_path:
        return "rag_sequence"
    if "bart" in model_name_or_path:
        return "bart"
    return None


model_kwargs = {}
if args.model_type is None:
    args.model_type = infer_model_type(args.model_name_or_path)
    assert args.model_type is not None
if args.model_type.startswith("rag"):
    model_class = RagTokenForGeneration if args.model_type == "rag_token" else RagSequenceForGeneration
    ragRetrieverClass = RagRetriever
    model_kwargs["n_docs"] = args.n_docs
    if args.index_name is not None:
        model_kwargs["index_name"] = args.index_name
    if args.index_path is not None:
        model_kwargs["index_path"] = args.index_path
elif args.model_type.startswith("confidential_rag"):
    model_class = ConfidentialRagSequenceForGeneration
    ragRetrieverClass = ConfidentialRagRetriever
    model_kwargs["n_docs"] = args.n_docs
    if args.index_name is not None:
        model_kwargs["index_name"] = args.index_name
    if args.index_path is not None:
        model_kwargs["index_path"] = args.index_path
else:
    model_class = BartForConditionalGeneration

checkpoint = args.model_name_or_path

logger.info("Evaluate the following checkpoints: %s", checkpoint)


logger.info("***** Running evaluation for {} *****".format(checkpoint))
logger.info("  Batch size = %d", args.eval_batch_size)
logger.info("  Predictions will be stored under {}".format(args.predictions_path))

if args.model_type.startswith("rag") or args.model_type.startswith("confidential_rag"):
    retriever = ragRetrieverClass.from_pretrained(checkpoint, **model_kwargs)
    model = model_class.from_pretrained(checkpoint, retriever=retriever, **model_kwargs)
    model.retriever.init_retrieval()
    if args.numConfidential is not None:
        # retriever.numConfidential = args.numConfidential
        model.numConfidential = args.numConfidential
        model.rag.numConfidential = args.numConfidential
    if args.generationMode is not None:
        model.generationMode = args.generationMode
else:
    model = model_class.from_pretrained(checkpoint, **model_kwargs)
model.to(args.device)

Popen(['git', 'version'], cwd=/home/huahua/Projects/transformers-3.3.1/examples/rag, stdin=None, shell=False, universal_newlines=False)
Popen(['git', 'version'], cwd=/home/huahua/Projects/transformers-3.3.1/examples/rag, stdin=None, shell=False, universal_newlines=False)
Evaluate the following checkpoints: /home/huahua/Projects/transformers-3.3.1/models/rag-sequence-nq
***** Running evaluation for /home/huahua/Projects/transformers-3.3.1/models/rag-sequence-nq *****
  Batch size = 4
  Predictions will be stored under /home/huahua/Projects/transformers-3.3.1/examples/rag/output/e2e_preds.txt
loading configuration file /home/huahua/Projects/transformers-3.3.1/models/rag-sequence-nq/config.json
Model config RagConfig {
  "architectures": [
    "RagSequenceForGeneration"
  ],
  "bad_words_ids": [
    [
      0,
      0
    ]
  ],
  "dataset": "wiki_dpr",
  "dataset_split": "train",
  "do_deduplication": true,
  "do_marginalize": false,
  "doc_sep": " // ",
  "exclude_bos_score": false,
  "

ConfidentialRagSequenceForGeneration(
  (rag): ConfidentialRagModel(
    (question_encoder): DPRQuestionEncoder(
      (question_encoder): DPREncoder(
        (bert_model): BertModel(
          (embeddings): BertEmbeddings(
            (word_embeddings): Embedding(30522, 768, padding_idx=0)
            (position_embeddings): Embedding(512, 768)
            (token_type_embeddings): Embedding(2, 768)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): BertEncoder(
            (layer): ModuleList(
              (0-11): 12 x BertLayer(
                (attention): BertAttention(
                  (self): BertSelfAttention(
                    (query): Linear(in_features=768, out_features=768, bias=True)
                    (key): Linear(in_features=768, out_features=768, bias=True)
                    (value): Linear(in_features=768, out_features=768, bias=True)
                

In [9]:
# Original Model
class OriginalRagRetriever(RagRetriever):
    def __init__(
        self,
        config,
        question_encoder_tokenizer,
        generator_tokenizer,
        index=None,
        init_retrieval=True,
    ):
        self.question_encoder_tokenizer = question_encoder_tokenizer
        self.generator_tokenizer = generator_tokenizer
        self.index = index

        self.config = config
        self.n_docs = config.n_docs
        self.batch_size = config.retrieval_batch_size


originalRetriever = OriginalRagRetriever(
    config=retriever.config,
    question_encoder_tokenizer=retriever.question_encoder_tokenizer,
    generator_tokenizer=retriever.generator_tokenizer,
    index=retriever.index,
    init_retrieval=False,
)
originalModel = RagSequenceForGeneration(
    config=model.config,
    question_encoder=model.question_encoder,
    generator=model.generator,
    retriever=originalRetriever,
)

## Evaluation

### Prepare Dataset

In [10]:
# Dataset
import random


def evaluationDataLoader(ratio=0.1, shuffle=True):
    with open(args.evaluation_set, "r") as eval_file, open(
        args.gold_data_path, "r"
    ) as gold_file:
        questions = eval_file.readlines()
        answers = gold_file.readlines()
        assert len(questions) == len(answers), "Question and Answer size must be equal."
        siz = len(questions)
        qa = [(questions[i], answers[i]) for i in range(siz)]
        if shuffle:
            random.shuffle(qa)
        xy = qa[: int(siz * ratio)]
        x = [item[0] for item in xy]
        y = [item[1] for item in xy]
        return x, y

In [11]:
# Evalutation Function


def metric_max_over_ground_truths(metric_fn, prediction, ground_truths):
    return max(metric_fn(prediction, gt) for gt in ground_truths)


def get_scores(args, preds, labels):
    f1 = em = total = 0
    labels = [[item.strip()] for item in labels]
    for prediction, ground_truths in zip(preds, labels):
        total += 1
        em += metric_max_over_ground_truths(
            exact_match_score, prediction, ground_truths
        )
        f1 += metric_max_over_ground_truths(f1_score, prediction, ground_truths)

    em = 100.0 * em / total
    f1 = 100.0 * f1 / total

    logger.info(f"F1: {f1:.2f}")
    logger.info(f"EM: {em:.2f}")


def get_precision_at_k(args, preds_path, gold_data_path):
    k = args.k
    hypos = [line.strip() for line in open(preds_path, "r").readlines()]
    references = [line.strip() for line in open(gold_data_path, "r").readlines()]

    em = total = 0
    for hypo, reference in zip(hypos, references):
        hypo_provenance = set(hypo.split("\t")[:k])
        ref_provenance = set(reference.split("\t"))
        total += 1
        em += len(hypo_provenance & ref_provenance) / k

    em = 100.0 * em / total
    logger.info(f"Precision@{k}: {em: .2f}")


def evaluate_batch_retrieval(args, rag_model, questions):
    def strip_title(title):
        if title.startswith('"'):
            title = title[1:]
        if title.endswith('"'):
            title = title[:-1]
        return title

    retriever_input_ids = (
        rag_model.retriever.question_encoder_tokenizer.batch_encode_plus(
            questions,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )["input_ids"].to(args.device)
    )

    question_enc_outputs = rag_model.rag.question_encoder(retriever_input_ids)
    question_enc_pool_output = question_enc_outputs[0]

    result = rag_model.retriever(
        retriever_input_ids,
        question_enc_pool_output.cpu().detach().to(torch.float32).numpy(),
        prefix=rag_model.rag.generator.config.prefix,
        n_docs=rag_model.config.n_docs,
        return_tensors="pt",
    )
    all_docs = rag_model.retriever.index.get_doc_dicts(result.doc_ids)
    provenance_strings = []
    for docs in all_docs:
        provenance = [strip_title(title) for title in docs["title"]]
        provenance_strings.append("\t".join(provenance))
    return provenance_strings


def evaluate_batch_e2e(args, rag_model, questions):
    with torch.no_grad():
        inputs_dict = rag_model.retriever.question_encoder_tokenizer.batch_encode_plus(
            questions, return_tensors="pt", padding=True, truncation=True
        )

        input_ids = inputs_dict.input_ids.to(args.device)
        attention_mask = inputs_dict.attention_mask.to(args.device)
        outputs = rag_model.generate(  # rag_model overwrites generate
            input_ids,
            attention_mask=attention_mask,
            num_beams=args.num_beams,
            min_length=args.min_length,
            max_length=args.max_length,
            early_stopping=False,
            num_return_sequences=1,
            bad_words_ids=[
                [0, 0]
            ],  # BART likes to repeat BOS tokens, dont allow it to generate more than one
        )
        answers = rag_model.retriever.generator_tokenizer.batch_decode(
            outputs, skip_special_tokens=True
        )

        return answers


def eval(model, x, y):
    score_fn = get_scores if args.eval_mode == "e2e" else get_precision_at_k
    evaluate_batch_fn = (
        evaluate_batch_e2e if args.eval_mode == "e2e" else evaluate_batch_retrieval
    )
    with open(args.predictions_path, "w") as preds_file:
        preds = []

        def doEval(questions, labels):
            answers = evaluate_batch_fn(args, model, questions)
            preds.extend(answers)
            preds_file.write("\n".join(answers) + "\n")
            preds_file.flush()
            if args.print_predictions:
                for q, a, y in zip(questions, answers, labels):
                    logger.info("\nQ: {}\nA: {}\nY: {}".format(q, a, y))

        questions = []
        labels = []
        for q, label in tqdm(zip(x, y)):
            questions.append(q.strip())
            labels.append(label)
            if len(questions) == args.eval_batch_size:
                doEval(questions, labels)
                questions = []
                labels = []
        if len(questions) > 0:
            doEval(questions, labels)

        score_fn(args, preds, y)

In [12]:
# Evaluation Data
X, Y = evaluationDataLoader(0.001, shuffle=False)

### For Original Evaluation

In [13]:
# Normal Eval
eval(originalModel, X, Y)

0it [00:00, ?it/s]/home/huahua/Projects/transformers-3.3.1/src/transformers/tokenization_utils_base.py:555: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor = as_tensor(value)

Q: who sings does he love me with reba
A:  linda kaye davis
Y: Linda Davis


Q: where do the great lakes meet the ocean
A:  saint lawrence river
Y: the Saint Lawrence River


Q: when does the new my hero academia movie come out
A:  october 17, 2018
Y: July 5 , 2018


Q: who was the creator of victoria 's secret
A:  roy raymond
Y: Roy Raymond

6it [00:22,  3.81s/it]
/home/huahua/Projects/transformers-3.3.1/src/transformers/generation_utils.py:948: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/ten

### For Normal/Confidential Model

In [14]:
# update method if need
enableUpdate = 1 == 1
if enableUpdate:
    model.rag.retriever.postprocess_docs = (
        ConfidentialRagRetriever.postprocess_docs.__get__(
            model.rag.retriever, ConfidentialRagRetriever
        )
    )
    model.rag.retriever.concatConfidentialDocs = (
        ConfidentialRagRetriever.concatConfidentialDocs.__get__(
            model.rag.retriever, ConfidentialRagRetriever
        )
    )
    model.rag.retriever.__call__ = ConfidentialRagRetriever.__call__.__get__(
        model.rag.retriever, ConfidentialRagRetriever
    )
    model.rag.forward = ConfidentialRagModel.forward.__get__(
        model, ConfidentialRagModel
    )
    model.generate = ConfidentialRagSequenceForGeneration.generate.__get__(
        model, ConfidentialRagSequenceForGeneration
    )

In [1]:
if args.numConfidential is not None:
    nc = 5
    model.numConfidential = nc
    model.rag.numConfidential = nc

eval(model, X, Y)

NameError: name 'args' is not defined